In [6]:
import torch

In [7]:
def cut(W,p):
    return ((W @ p) * (1-p)).sum((1, 2))

def S(p):
    return -(p*p.log()).sum(2).sum(1)

def argmax_cut(W,p):
    s = torch.nn.functional.one_hot(p.argmax(dim=2), num_classes=p.shape[2])
    return config, cut(W, s) / 2

In [3]:
def balance(p):
    return (p.sum(1)**2).sum(1)-(p**2).sum(2).sum(1)

In [5]:
def solve(problem,W,batch,q,panelty,beta_range):
    n = W.shape[0]
    h = torch.rand(batch, n, q)
    optimizer = torch.optim.Adam([h], lr=0.01)
    
    for beta in beta_range:
        p = torch.softmax(h, dim=2)
        if problem == "maxcut":
            F = -cut(W,p) - S(p)/beta
        if problem == "bmincut":
            F = cut(W,p)+panelty*balance(p)-S(p)/beta
        optimizer.zero_grad()
        F.backward(gradient=torch.ones_like(F))
        optimizer.step()
    
    return argmax_cut(W, p)